# Tariikhna — Upload Fine-Tuned Model to Hugging Face Hub

Pushes your trained LoRA adapters to HF Hub for storage, versioning, and sharing.
This makes your model citable and reproducible (good for your graduation document).

### Before running:
1. Create a free account at https://huggingface.co
2. Create an access token: Settings -> Access Tokens -> New token (role: **Write**)
3. Have your `tariikhna_lora.zip` (the adapters you downloaded from training)

No GPU needed for this - runs on a free CPU runtime.

In [ ]:
!pip install huggingface_hub -q

## 1. Upload Your Adapters Zip

In [ ]:
# Upload the tariikhna_lora.zip you downloaded from training
from google.colab import files
print('Upload your tariikhna_lora.zip:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

In [ ]:
# Unzip it
import zipfile, os
os.makedirs('tariikhna_lora', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('tariikhna_lora')

print('Extracted files:')
for f in os.listdir('tariikhna_lora'):
    size = os.path.getsize(f'tariikhna_lora/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')

## 2. Log In to Hugging Face

In [ ]:
from huggingface_hub import login
# Paste your WRITE token when prompted
login()

## 3. Push to Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

# ============================================
# SET YOUR REPO NAME (username/model-name)
# ============================================
HF_USERNAME = 'your-username'   # <- your HF username
MODEL_NAME = 'tariikhna-llama-3.1-8b-lora'
REPO_ID = f'{HF_USERNAME}/{MODEL_NAME}'

# Create the repo (private by default - safer for a student project)
create_repo(REPO_ID, private=True, exist_ok=True)
print(f'Repo created: {REPO_ID}')

# Upload all adapter files
api = HfApi()
api.upload_folder(
    folder_path='tariikhna_lora',
    repo_id=REPO_ID,
    repo_type='model',
)
print(f'\nUploaded! View at: https://huggingface.co/{REPO_ID}')

## 4. Add a Model Card (good for your document)

A README that describes your model - shows professionalism in your thesis.

In [ ]:
model_card = f"""---
base_model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
library_name: peft
tags:
- unsloth
- lora
- islamic-education
- children-content
---

# Tariikhna - Llama 3.1 8B LoRA

Fine-tuned LoRA adapters for the Tariikhna project: an AI-powered Islamic
educational comic platform for children (ages 6-12).

## Purpose
Converts a single visual scene (from a segmented Sira passage) into a structured
comic-panel schema including a detailed image-generation prompt, following Islamic
depiction guidelines (prophets shown from behind, modest dress, child-appropriate content).

## Base Model
Meta Llama 3.1 8B Instruct (4-bit quantized), fine-tuned with QLoRA via Unsloth.

## Training
- Method: QLoRA (rank 16, all linear projections)
- Trainable params: ~42M (0.52% of 8B)
- Dataset: scene -> schema pairs generated via a multi-layer pipeline from Sira text

## Usage
Load with the base model and Unsloth FastLanguageModel - see project repository.

## Academic Project
Graduation Project, Faculty of Computers and Artificial Intelligence, Cairo University.
"""

with open('README.md', 'w') as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj='README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type='model',
)
print('Model card uploaded!')
print(f'View at: https://huggingface.co/{REPO_ID}')